# Neural Network Embedding Recommender and Model Comparison

This notebook trains a neural recommender using user and course embeddings, then compares the collaborative-filtering results used in the capstone presentation.

## Dependency note

This notebook uses TensorFlow/Keras and scikit-learn. Install the requirements before running it. Neural-network RMSE can vary slightly unless the environment, seed, split, and hardware are identical.

In [ ]:
from pathlib import Path
import urllib.request
import pandas as pd
import numpy as np

DATA_DIR = Path("datasets")
DATA_URLS = {
    "ratings.csv": "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-ML321EN-SkillsNetwork/labs/datasets/ratings.csv",
    "course_genre.csv": "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-ML321EN-SkillsNetwork/labs/datasets/course_genre.csv",
    "rs_content_test.csv": "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-ML321EN-SkillsNetwork/labs/datasets/rs_content_test.csv",
    "user_profile.csv": "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-ML321EN-SkillsNetwork/labs/datasets/user_profile.csv",
    "course_processed.csv": "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-ML321EN-SkillsNetwork/labs/datasets/course_processed.csv",
    "courses_bows.csv": "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-ML321EN-SkillsNetwork/labs/datasets/courses_bows.csv",
    "sim.csv": "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-ML321EN-SkillsNetwork/labs/datasets/sim.csv",
}

def ensure_dataset(filename):
    DATA_DIR.mkdir(exist_ok=True)
    path = DATA_DIR / filename
    if not path.exists():
        print(f"Downloading {filename}...")
        urllib.request.urlretrieve(DATA_URLS[filename], path)
    return path

def load_csv(filename, **kwargs):
    return pd.read_csv(ensure_dataset(filename), **kwargs)

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 120)

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

np.random.seed(42)
tf.random.set_seed(42)

ratings_df = load_csv("ratings.csv")
ratings_df.head()

In [ ]:
user_encoder = LabelEncoder()
item_encoder = LabelEncoder()

ratings_model_df = ratings_df.copy()
ratings_model_df["user_idx"] = user_encoder.fit_transform(ratings_model_df["user"])
ratings_model_df["item_idx"] = item_encoder.fit_transform(ratings_model_df["item"])

x = ratings_model_df[["user_idx", "item_idx"]].to_numpy()
y = (ratings_model_df["rating"].to_numpy(dtype=float) - 2.0)

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.20, random_state=42)
x_train, x_val, y_train, y_val = train_test_split(x_train, y_train, test_size=0.10, random_state=42)

num_users = ratings_model_df["user_idx"].nunique()
num_items = ratings_model_df["item_idx"].nunique()

print("Encoded users:", num_users)
print("Encoded courses:", num_items)
print("Train/validation/test sizes:", len(x_train), len(x_val), len(x_test))

In [ ]:
class RecommenderNet(keras.Model):
    def __init__(self, num_users, num_items, embedding_size=16):
        super().__init__()
        self.user_embedding = layers.Embedding(num_users, embedding_size)
        self.item_embedding = layers.Embedding(num_items, embedding_size)
        self.user_bias = layers.Embedding(num_users, 1)
        self.item_bias = layers.Embedding(num_items, 1)
        self.output_layer = layers.Dense(1, activation="sigmoid")

    def call(self, inputs):
        user_vector = self.user_embedding(inputs[:, 0])
        item_vector = self.item_embedding(inputs[:, 1])
        dot_product = tf.reduce_sum(user_vector * item_vector, axis=1, keepdims=True)
        x = dot_product + self.user_bias(inputs[:, 0]) + self.item_bias(inputs[:, 1])
        return self.output_layer(x)

model = RecommenderNet(num_users, num_items, embedding_size=16)
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss=keras.losses.MeanSquaredError(),
    metrics=[keras.metrics.RootMeanSquaredError(name="rmse")],
)

history = model.fit(
    x_train,
    y_train,
    validation_data=(x_val, y_val),
    epochs=10,
    batch_size=64,
    verbose=1,
)

In [ ]:
test_loss, test_rmse = model.evaluate(x_test, y_test, verbose=1)
print(f"Neural Network Test RMSE: {test_rmse:.4f}")

## Collaborative Filtering Comparison

The presentation reports these final RMSE values. Lower RMSE means better rating prediction.

In [ ]:
reported_results = pd.DataFrame(
    {
        "model": ["KNN", "NMF", "Neural Network Embedding"],
        "rmse": [0.2063, 0.2048, 0.1534],
        "relative_training_cost": ["Low", "Medium", "High"],
    }
)

display(reported_results)

try:
    import matplotlib.pyplot as plt

    plt.figure(figsize=(8, 5))
    plt.bar(reported_results["model"], reported_results["rmse"], color=["#4C78A8", "#59A14F", "#F28E2B"])
    plt.ylabel("RMSE")
    plt.title("Collaborative Filtering Model Comparison")
    plt.ylim(0, 0.25)
    for idx, value in enumerate(reported_results["rmse"]):
        plt.text(idx, value + 0.005, f"{value:.4f}", ha="center")
    plt.tight_layout()
    plt.show()
except ImportError:
    print("Install matplotlib to display the chart.")

## Interpretation

The neural embedding model is the strongest model in the presentation because it learns dense user and course representations and can capture non-linear user-course preference patterns. A practical production system would combine content-based methods for cold-start users with neural collaborative filtering for users who have enough interaction history.